# Fix `age_Myr` for the target object

This notebook finds CSV files in the top-level `Code` directory and in `makingtable/`, identifies rows for the target object using any available identifier column, and updates `age_Myr` from `112` to `125` only for those rows.

The workflow is split into two parts:
1. A dry-run preview that reports what would change.
2. A write step that is disabled by default until you review the notebook.

In [1]:
from pathlib import Path
import csv
from collections import defaultdict

# Define the exact identifier values for this fix.
# A row is considered a match if any of these values appear in the corresponding column,
# and that column exists in the CSV being checked.
TARGET_TIC = "440725886"
TARGET_POP_ID = "124"
TARGET_GAIA_ID = "63661521387693824"
OLD_AGE_VALUES = {"112", "112.0"}
NEW_AGE_VALUE = "125"

# Limit the search to the top-level Code directory and the makingtable directory.
# This intentionally skips other subdirectories such as SRMP.
SEARCH_DIRS = [Path('.'), Path('makingtable')]


def candidate_csv_files():
    """Return the CSV files that should be checked for this fix."""
    files = []
    for directory in SEARCH_DIRS:
        files.extend(sorted(directory.glob('*.csv')))
    return files


def sniff_dialect_and_newline(path):
    """Detect the CSV dialect and line ending so we can preserve file style when rewriting."""
    with path.open('r', encoding='utf-8-sig', newline='') as handle:
        raw_text = handle.read()

    if '\r\n' in raw_text:
        newline = '\r\n'
    elif '\r' in raw_text:
        newline = '\r'
    else:
        newline = '\n'

    sample = raw_text[:4096]
    try:
        dialect = csv.Sniffer().sniff(sample)
    except csv.Error:
        dialect = csv.excel

    return dialect, newline


def row_matches_target(row, fieldnames):
    """Return True if the row matches the target object using any available identifier column."""
    if 'TIC' in fieldnames and row.get('TIC') == TARGET_TIC:
        return True
    if 'pop_id' in fieldnames and row.get('pop_id') == TARGET_POP_ID:
        return True
    if 'gaiadr3_source_id' in fieldnames and row.get('gaiadr3_source_id') == TARGET_GAIA_ID:
        return True
    return False


def inspect_file(path):
    """Read a CSV file and summarize whether it contains rows that need updating."""
    with path.open('r', encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames or []

        has_age = 'age_Myr' in fieldnames
        available_id_columns = [
            column for column in ('TIC', 'pop_id', 'gaiadr3_source_id') if column in fieldnames
        ]
        matched_rows = []
        rows_to_update = 0

        if available_id_columns and has_age:
            for row_number, row in enumerate(reader, start=2):
                if row_matches_target(row, fieldnames):
                    matched_rows.append((row_number, row.get('age_Myr')))
                    if row.get('age_Myr') in OLD_AGE_VALUES:
                        rows_to_update += 1

    return {
        'path': path,
        'has_age': has_age,
        'available_id_columns': available_id_columns,
        'matched_rows': matched_rows,
        'rows_to_update': rows_to_update,
    }


# Build an inventory we can reuse in the preview and write steps.
file_summaries = [inspect_file(path) for path in candidate_csv_files()]
target_summaries = [
    summary for summary in file_summaries
    if summary['has_age'] and summary['available_id_columns'] and summary['matched_rows']
]

print(f"Checked {len(file_summaries)} CSV files.")
print(f"Found {len(target_summaries)} files with 'age_Myr' and at least one matching target row.")

Checked 24 CSV files.
Found 15 files with 'age_Myr' and at least one matching target row.


In [2]:
# Dry-run preview: show exactly which files and rows would be updated.
total_rows_to_update = 0

for summary in target_summaries:
    age_values = sorted({age for _, age in summary['matched_rows']})
    total_rows_to_update += summary['rows_to_update']
    print(summary['path'])
    print(f"  identifier columns used: {summary['available_id_columns']}")
    print(f"  matching target rows: {len(summary['matched_rows'])}")
    print(f"  current age_Myr values on those rows: {age_values}")
    print(f"  rows that would be changed from 112 to 125: {summary['rows_to_update']}")

print()
print(f"Total rows that would be updated: {total_rows_to_update}")

lcnameswithstatus.csv
  identifier columns used: ['TIC', 'pop_id', 'gaiadr3_source_id']
  matching target rows: 5
  current age_Myr values on those rows: ['112.0']
  rows that would be changed from 112 to 125: 5
lcscoresandchecker.csv
  identifier columns used: ['TIC', 'pop_id', 'gaiadr3_source_id']
  matching target rows: 5
  current age_Myr values on those rows: ['112']
  rows that would be changed from 112 to 125: 5
lcscoresreformat.csv
  identifier columns used: ['TIC', 'pop_id', 'gaiadr3_source_id']
  matching target rows: 5
  current age_Myr values on those rows: ['112.0']
  rows that would be changed from 112 to 125: 5
lcscoreswgaia.csv
  identifier columns used: ['TIC', 'pop_id', 'gaiadr3_source_id']
  matching target rows: 1
  current age_Myr values on those rows: ['112.0']
  rows that would be changed from 112 to 125: 1
periodtable.csv
  identifier columns used: ['TIC', 'pop_id', 'gaiadr3_source_id']
  matching target rows: 5
  current age_Myr values on those rows: ['112.0']


In [4]:
# Write step: leave WRITE_CHANGES as False while reviewing.
# Change it to True only when you are ready to apply the fix.
WRITE_CHANGES = True

if not WRITE_CHANGES:
    raise RuntimeError("WRITE_CHANGES is False. Review the dry-run output first, then set WRITE_CHANGES = True to apply the update.")


def update_file(path):
    """Update only the target rows in one CSV file and preserve the existing CSV style as closely as possible."""
    dialect, newline = sniff_dialect_and_newline(path)

    with path.open('r', encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle, dialect=dialect)
        fieldnames = reader.fieldnames or []
        rows = list(reader)

    if 'age_Myr' not in fieldnames:
        return 0

    available_id_columns = [
        column for column in ('TIC', 'pop_id', 'gaiadr3_source_id') if column in fieldnames
    ]
    if not available_id_columns:
        return 0

    updated_count = 0
    for row in rows:
        if row_matches_target(row, fieldnames) and row.get('age_Myr') in OLD_AGE_VALUES:
            row['age_Myr'] = NEW_AGE_VALUE
            updated_count += 1

    if updated_count == 0:
        return 0

    with path.open('w', encoding='utf-8-sig', newline='') as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=fieldnames,
            dialect=dialect,
            lineterminator=newline,
        )
        writer.writeheader()
        writer.writerows(rows)

    return updated_count


# Apply the change only to files that the preview identified.
updated_by_file = {}
for summary in target_summaries:
    updated_rows = update_file(summary['path'])
    if updated_rows:
        updated_by_file[str(summary['path'])] = updated_rows

print('Updated files:')
for path, count in updated_by_file.items():
    print(f"  {path}: {count} rows updated")

print()
print(f"Total rows updated: {sum(updated_by_file.values())}")

Updated files:
  lcnameswithstatus.csv: 5 rows updated
  lcscoresandchecker.csv: 5 rows updated
  lcscoresreformat.csv: 5 rows updated
  periodtable.csv: 5 rows updated
  makingtable/Full_complex_sarah - Objects and References.csv: 1 rows updated
  makingtable/availablelightcurves.csv: 1 rows updated
  makingtable/bestlcs.csv: 1 rows updated
  makingtable/idswithdiscopaper.csv: 1 rows updated
  makingtable/ourgaiaids.csv: 1 rows updated
  makingtable/papertable2.3.csv: 1 rows updated
  makingtable/papertable5.csv: 5 rows updated

Total rows updated: 31
